# Checkpoint B: ejecución del protocolo congelado

Este cuaderno solo lee los resultados de `run_checkpoint_b_frozen.py`. Las conclusiones son condicionales al supuesto no verificado de disponibilidad antes de abrir `t+1`; no hay HMM ni integración Actinver.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd().resolve().parent
if not (ROOT/'reports').exists(): ROOT=Path.cwd().resolve()
R=ROOT/'reports'
summary=pd.read_csv(R/'checkpoint_b_partition_report.csv')
daily=pd.read_csv(R/'checkpoint_b_daily_results.csv',parse_dates=['signal_date','decision','breach'])
episodes=pd.read_csv(R/'checkpoint_b_episodes.csv',parse_dates=['start','end'])
display(summary)

## Verificaciones antes del desempeño

El script ejecuta pruebas sintéticas de: etiqueta con brecha futura, agrupación a cinco sesiones, asignación al primer episodio y exclusión por frontera.

**Límite.** Estas pruebas validan mecánica, no la validez económica del evento.

In [ ]:
# Ejemplos concretos de etiqueta y episodio generados por la ejecución
display(daily.loc[daily['label'].eq(1), ['signal_date','decision','label','breach','skew','zskew']].head(5))
display(episodes.head(8))

## Alertas, asignación y cobertura

Una alerta se asigna como máximo al primer episodio que inicia en las cinco decisiones posteriores. Fechas sin skew permanecen en el denominador de cobertura y equivalen a ausencia de alerta en una ventana de episodio.

**Límite.** La evaluación es de la definición congelada de caída de SPY, no de regímenes generales.

In [ ]:
cols=['signal_date','part','bench_alert','skew_alert','wide','roll_or_jump']
display(daily.loc[(daily['bench_alert']) | (daily['skew_alert']), cols].head(12))
display(summary[['partition','coverage_denominator','skew_available','skew_missing','common_dates','episodes_evaluable','incomplete_episode_windows','decision']])

## Calidad y sensibilidad

La sensibilidad separada suprime alertas de skew en fechas con `wide_spread`, salto extremo o cambio de vencimiento; no elimina etiquetas ni redefine episodios.

**Límite.** Es una sensibilidad descriptiva y no puede reemplazar el resultado primario.

In [ ]:
sens=pd.read_csv(R/'checkpoint_b_quality_sensitivity_daily.csv')
display(sens.groupby('part')[['wide','roll_or_jump','skew_alert']].sum())

## Decisión

La muestra final tiene menos de diez episodios evaluables, por lo que el estado obligatorio es **INCONCLUSO**. Métricas secundarias no pueden convertirlo en CANDIDATO ni STOP. Antes de cualquier ampliación, se debe mantener visible la dependencia del supuesto de disponibilidad de `t+1`.